In [11]:
from collections import Counter
import os
import re

import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

import nltk
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer, WordNetLemmatizer

from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
import re

In [12]:
nltk.download('stopwords')
nltk.download('wordnet')

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\diego\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\diego\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


True

In [13]:
from pathlib import Path

import pandas as pd

def load_imdb_data(path):
    path = Path(path)
    data = []
    labels = []

    for label in ['pos', 'neg']:
        dir_path = path / label

        for file_path in sorted(dir_path.glob('*.txt')):
            with open(file_path, 'r', encoding='utf-8') as file:
                data.append(file.read())
                labels.append(1 if label == 'pos' else 0)

    return pd.DataFrame({'review': data, 'sentiment': labels})


dataset_root = Path('data/aclImdb_v1/aclImdb')

train_df = load_imdb_data(dataset_root / 'train')
test_df = load_imdb_data(dataset_root / 'test')

In [14]:
english_stopwords = stopwords.words('english')

neg_words = [
    'no', 'nor', 'not', 'ain', 'aren', "aren't", 'don', "don't",
    'couldn', "couldn't", 'didn', "didn't", 'doesn', "doesn't",
    'hadn', "hadn't", 'hasn', "hasn't", 'haven', "haven't",
    'isn', "isn't", 'mightn', "mightn't", 'mustn', "mustn't",
    'needn', "needn't", 'shan', "shan't", 'shouldn', "shouldn't",
    'wasn', "wasn't", 'weren', "weren't", 'won', "won't",
    'wouldn', "wouldn't"
]

english_stopwords = [w for w in english_stopwords if w not in neg_words]


def normalize_text(text):
    text = text.lower()
    text = re.sub(r'[^a-z\s]', '', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

In [15]:
stemmer = PorterStemmer()
lemmatizer = WordNetLemmatizer()


def preprocess_stem(text):
    text = normalize_text(text)
    return ' '.join([stemmer.stem(w) for w in text.split()])


def preprocess_lemma(text):
    text = normalize_text(text)
    return ' '.join([lemmatizer.lemmatize(w) for w in text.split()])

In [16]:
RANDOM_STATE = 42

train_exp_df = train_df.sample(frac=1, random_state=RANDOM_STATE).reset_index(drop=True)
test_exp_df = test_df.sample(frac=1, random_state=RANDOM_STATE).reset_index(drop=True)

In [17]:
def run_experiment(vectorizer, preprocess_fn=None):
    train_text = train_exp_df['review']
    test_text = test_exp_df['review']

    if preprocess_fn is not None:
        train_text = train_text.apply(preprocess_fn)
        test_text = test_text.apply(preprocess_fn)

    x_train_vec = vectorizer.fit_transform(train_text)
    y_train = train_exp_df['sentiment']

    x_test_vec = vectorizer.transform(test_text)
    y_test = test_exp_df['sentiment']

    model = LogisticRegression(
        solver='liblinear',
        max_iter=1000,
        random_state=RANDOM_STATE
    )

    model.fit(x_train_vec, y_train)

    y_train_pred = model.predict(x_train_vec)
    y_test_pred = model.predict(x_test_vec)

    metrics = {
        'accuracy_train': accuracy_score(y_train, y_train_pred),
        'accuracy_test': accuracy_score(y_test, y_test_pred),
        'precision_train': precision_score(y_train, y_train_pred),
        'precision_test': precision_score(y_test, y_test_pred),
        'recall_train': recall_score(y_train, y_train_pred),
        'recall_test': recall_score(y_test, y_test_pred),
        'f1_train': f1_score(y_train, y_train_pred),
        'f1_test': f1_score(y_test, y_test_pred),
    }

    return model, vectorizer, metrics

In [18]:
experiments = [
    ('CountVectorizer-stem', preprocess_stem, CountVectorizer(stop_words=english_stopwords, lowercase=True)),
    ('CountVectorizer-lemma', preprocess_lemma, CountVectorizer(stop_words=english_stopwords, lowercase=True)),
    ('TfidfVectorizer-stem', preprocess_stem, TfidfVectorizer(stop_words=english_stopwords, lowercase=True)),
    ('TfidfVectorizer-lemma', preprocess_lemma, TfidfVectorizer(stop_words=english_stopwords, lowercase=True)),
]

results = []
model_runs = []

for exp_name, preprocess_fn, vectorizer in experiments:
    model, fitted_vectorizer, metrics = run_experiment(vectorizer, preprocess_fn=preprocess_fn)

    results.append({
        'vectorizer': exp_name,
        **metrics,
    })

    model_runs.append({
        'name': exp_name,
        'model': model,
        'fitted_vectorizer': fitted_vectorizer,
    })

results_df = pd.DataFrame(results).sort_values('accuracy_test', ascending=False)
results_df

,vectorizer,accuracy_train,accuracy_test,precision_train,precision_test,recall_train,recall_test,f1_train,f1_test
3,TfidfVectorizer-lemma,0.93456,0.88084,0.928054,0.881114,0.94216,0.88048,0.935054,0.880797
2,TfidfVectorizer-stem,0.92900,0.87800,0.922144,0.877758,0.93712,0.87832,0.929572,0.878039
1,CountVectorizer-lemma,0.99844,0.85992,0.998639,0.866428,0.99824,0.85104,0.998440,0.858665
0,CountVectorizer-stem,0.99676,0.85536,0.997118,0.863562,0.99640,0.84408,0.996759,0.853710


In [19]:
def get_top_features(model, fitted_vectorizer, top_n=10):
    feature_names = fitted_vectorizer.get_feature_names_out()
    coefs = model.coef_[0]

    top_pos_idx = coefs.argsort()[::-1][:top_n]
    top_neg_idx = coefs.argsort()[:top_n]

    top_positive = [(feature_names[idx], coefs[idx]) for idx in top_pos_idx]
    top_negative = [(feature_names[idx], coefs[idx]) for idx in top_neg_idx]

    return top_positive, top_negative


def print_compact_feature_table(model_runs, sentiment='positive', top_n=10):
    table_data = {}

    for run in model_runs:
        top_positive, top_negative = get_top_features(
            run['model'],
            run['fitted_vectorizer'],
            top_n=top_n
        )

        selected = top_positive if sentiment == 'positive' else top_negative
        table_data[run['name']] = [f"{word} ({coef:.3f})" for word, coef in selected]

    compact_df = pd.DataFrame(table_data, index=[f"#{i}" for i in range(1, top_n + 1)])
    compact_df.index.name = 'rank'

    display(compact_df)


print('Top 10 positive words across experiments')
print_compact_feature_table(model_runs, sentiment='positive', top_n=10)

print('\nTop 10 negative words across experiments')
print_compact_feature_table(model_runs, sentiment='negative', top_n=10)

Top 10 positive words across experiments


,CountVectorizer-stem,CountVectorizer-lemma,TfidfVectorizer-stem,TfidfVectorizer-lemma
rank,,,,
#1,refresh (1.711),refreshing (1.730),great (7.127),great (7.211)
#2,flawless (1.599),wonderfully (1.458),excel (6.569),excellent (6.294)
#3,driven (1.491),funniest (1.442),love (5.098),best (5.090)
#4,superb (1.445),flawless (1.428),enjoy (5.078),perfect (4.754)
#5,excel (1.437),superb (1.397),best (4.887),wonderful (4.663)
#6,squirrel (1.412),excellent (1.329),perfect (4.865),favorite (4.645)
#7,erot (1.397),carrey (1.297),favorit (4.614),well (4.275)
#8,funniest (1.336),whoopi (1.293),well (4.072),amazing (4.216)
#9,whoopi (1.271),erotic (1.234),beauti (4.008),loved (4.098)



Top 10 negative words across experiments


,CountVectorizer-stem,CountVectorizer-lemma,TfidfVectorizer-stem,TfidfVectorizer-lemma
rank,,,,
#1,worst (-2.036),worst (-2.143),worst (-8.940),worst (-9.245)
#2,poorli (-1.858),disappointment (-2.107),wast (-7.372),bad (-7.448)
#3,mstk (-1.763),waste (-2.083),bad (-7.163),waste (-6.673)
#4,aw (-1.726),poorly (-1.805),aw (-6.595),awful (-6.461)
#5,wast (-1.627),awful (-1.694),bore (-6.040),boring (-5.476)
#6,unfunni (-1.549),disappointing (-1.515),poor (-5.338),poor (-5.322)
#7,forgett (-1.539),mstk (-1.487),disappoint (-4.688),nothing (-4.892)
#8,unwatch (-1.515),fails (-1.486),noth (-4.685),terrible (-4.751)
#9,alright (-1.501),laughable (-1.466),terribl (-4.666),worse (-4.736)


In [20]:
best_model_name = results_df.iloc[0]['vectorizer']
best_run = next(run for run in model_runs if run['name'] == best_model_name)

best_model = best_run['model']
best_vectorizer = best_run['fitted_vectorizer']

print("Best model:", best_model_name)
print("Best test accuracy:", results_df.iloc[0]['accuracy_test'])

Best model: TfidfVectorizer-lemma
Best test accuracy: 0.88084


In [21]:
my_review = "The movie was entertaining and well made. The story was interesting, the acting was strong, and I enjoyed watching it."

if 'lemma' in best_model_name:
    processed_review = preprocess_lemma(my_review)
else:
    processed_review = preprocess_stem(my_review)

review_vec = best_vectorizer.transform([processed_review])
prediction = best_model.predict(review_vec)[0]

print("Prediction:", "positive" if prediction == 1 else "negative")

Prediction: positive


### Which vectorizer and preprocessing combination gives the best performance?
El mejor modelo fue **TfidfVectorizer con lemmatization**, con un accuracy de prueba de **0.88084**. Esto muestra que TF-IDF funcionó mejor que CountVectorizer, y que lemmatization dio mejor resultado que stemming.

### Do you see any interesting differences in the top features?
En las top features se nota que TF-IDF usa palabras más claras para el sentimiento, como **great**, **excellent**, **best**, **worst**, **bad** y **awful**. En cambio, stemming corta las palabras y genera términos menos legibles como **excel**, **wast**, **aw** o **favorit**. Lemmatization mantiene palabras más naturales y fáciles de interpretar.

### Do you agree with the model's prediction?

El modelo predijo un sentimiento positivo. Estoy de acuerdo con la predicción porque la reseña usaba palabras positivas.

### Write your final remarks.

En conclusión, esta actividad muestra que el preprocesamiento sí afecta el desempeño del modelo. TF-IDF con lemmatization fue la mejor combinación porque conserva mejor el significado de las palabras y reduce el peso de términos demasiado comunes.